# Segment Anything Model (SAM) and Promptable Segmentation

The **Segment Anything Model (SAM)**, from Meta AI (2023), reframed segmentation as a **promptable** task: instead of training one model per dataset of fixed classes, SAM takes an image plus a **prompt** (points, a box, a rough mask, or text) and returns the mask of whatever the prompt indicates — **zero-shot**, on objects and domains it was never explicitly trained on.

Its power comes from scale: SAM was trained on **SA-1B**, ~1.1 billion masks over 11 million images, collected with a model-in-the-loop data engine. This makes it a strong, general-purpose, class-agnostic segmentation *foundation model* that downstream systems prompt rather than retrain.

## Architecture: three components

SAM separates expensive image understanding from cheap, interactive prompting:

```
              image                prompt (points / box / mask / text)
                |                              |
        +---------------+              +----------------+
        | Image Encoder |              | Prompt Encoder |
        |   (ViT, MAE-  |              | points/boxes-> |
        |   pretrained) |              | pos. embeds;   |
        +-------+-------+              | mask -> conv;  |
                |                      | text -> CLIP   |
         image embedding              +--------+-------+
                |                               |
                +---------- Mask Decoder -------+
                   (lightweight transformer)
                               |
                     masks + confidence (IoU) scores
```

1. **Image encoder** — a heavy **ViT** (Vision Transformer, MAE-pretrained) run **once** per image to produce a dense image embedding. This is the costly step.
2. **Prompt encoder** — encodes prompts: points/boxes as positional embeddings, a coarse mask via a small conv network, free-form text via a text/CLIP encoder.
3. **Mask decoder** — a **lightweight** transformer that fuses the image embedding with prompt embeddings and outputs masks. Because it is cheap, you can re-prompt the **same** cached image embedding interactively in real time.

## Key ideas

- **Ambiguity-aware output**: a single point can refer to nested objects (a shirt, a person, a crowd). SAM outputs **multiple candidate masks** with predicted **IoU/confidence scores** so the ambiguity is resolved by selection rather than guessing.
- **Decoupled cost**: encode the image once (expensive), then issue many prompts cheaply — ideal for interactive annotation tools.
- **Zero-shot transfer**: because it is class-agnostic and trained at massive scale, SAM generalizes to new domains (medical, satellite, microscopy) without fine-tuning, and composes with detectors that supply boxes.
- **Class-agnostic**: SAM segments *the region a prompt points to*; it does not, by itself, assign semantic class labels (that is delegated to the prompt source or a downstream classifier).

## Lineage: SAM -> SAM 2 -> the SAM 3 direction

- **SAM (2023)** — promptable image segmentation; ViT image encoder, prompt encoder, mask decoder; trained on SA-1B.
- **SAM 2 (2024)** — extends promptable segmentation to **video**, adding a **streaming memory** module so a mask prompted on one frame is tracked and propagated across frames (promptable visual segmentation in images *and* video) on the larger SA-V dataset.
- **"SAM 3" direction** — the evolving frontier of the family is **promptable *concept* segmentation**: prompting not just with a geometric click/box but with an open-vocabulary **concept** (a text phrase or example), and segmenting **all** instances matching that concept across an image or video. The aim is to unify *detect + segment + track* of an arbitrary concept under one promptable interface.

> Treat newer-than-SAM-2 capabilities as a **research direction**, not a fixed spec. Exact model names, APIs, and benchmark numbers for the latest releases evolve quickly — verify against the official Meta AI / `segment-anything` repositories before relying on specific class names or weights.

## Conceptual usage

The original SAM is distributed via Meta's `segment-anything` package. The canonical flow: build a predictor, **set the image once** (runs the heavy encoder), then prompt repeatedly.

```python
# Original SAM (segment-anything). Conceptual; check the repo for exact API & checkpoints.
from segment_anything import sam_model_registry, SamPredictor
import numpy as np

sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h.pth")
predictor = SamPredictor(sam)

predictor.set_image(image_rgb)              # run the ViT image encoder ONCE

# Prompt with a single foreground point (label 1 = foreground, 0 = background)
masks, scores, logits = predictor.predict(
    point_coords=np.array([[450, 300]]),
    point_labels=np.array([1]),
    multimask_output=True,                  # return several masks for ambiguity
)
best = masks[scores.argmax()]               # pick highest-confidence mask

# Or prompt with a bounding box instead:
# masks, scores, _ = predictor.predict(box=np.array([x0, y0, x1, y1]))
```

For fully automatic segmentation of *everything* in an image, the package also provides `SamAutomaticMaskGenerator`, which prompts SAM on a dense grid of points and returns all discovered masks. SAM 2 exposes an analogous predictor with additional video/memory methods.

## References

- Kirillov et al., *Segment Anything* (SAM), 2023 — https://arxiv.org/abs/2304.02643
- SAM project page & SA-1B dataset — https://segment-anything.com/
- `facebookresearch/segment-anything` (code & checkpoints) — https://github.com/facebookresearch/segment-anything
- Ravi et al., *SAM 2: Segment Anything in Images and Videos*, 2024 — https://arxiv.org/abs/2408.00714
- `facebookresearch/sam2` — https://github.com/facebookresearch/sam2
- Meta AI Segment Anything research hub (for the latest releases) — https://ai.meta.com/sam2/